In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm

import joblib
import utils

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
measure = "CMCT (msec)" # Regression target

In [ ]:
# Read in data
df = pd.read_csv(f"data/{measure}.csv")
df.head()

In [ ]:
# Standardise age, height and weight
scaler_height = StandardScaler()
scaler_age = StandardScaler()
scaler_weight = StandardScaler()

df["height_unadjusted"] = df["height"].copy()
df["age_unadjusted"] = df["age"].copy()
df["weight_unadjusted"] = df["weight"].copy()

df["height"] = scaler_height.fit_transform(df[["height"]])
df["age"] = scaler_age.fit_transform(df[["age"]])
df["weight"] = scaler_weight.fit_transform(df[["weight"]])

# Save scaler
joblib.dump(scaler_height, f"models/{measure}_scaler.joblib")

OLS setup

In [ ]:
cmct_string = "value ~ height + C(muscle) + height * C(muscle) "

ols_model = smf.ols(cmct_string, data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["patient_id"]}
)
ols_model.summary()

Residual variance over height

In [ ]:
plot_df = df.copy()
plot_df["muscle"] = np.where(plot_df["muscle"]=="Median (APB", "Median APB", plot_df["muscle"])
plot_df["prediction"] = ols_model.predict(df)
plot_df["resid"] = plot_df["value"] - plot_df["prediction"]

utils.residual_plot(plot_df)


# plt.savefig(
#     "paper_outputs/residual_diagnostic.png", dpi=600, bbox_inches="tight"
# )

plt.show()

Format and store regression coefficients 

In [ ]:
name_map = {
    "Intercept": "Intercept (Median APB as reference)",
    "C(muscle)[T.Median PT]": "Median PT (vs Median APB)",
    "C(muscle)[T.Peroneal TA]": "Peroneal TA (vs Median APB)",
    "C(muscle)[T.Tibial AH]": "Tibial AH (vs Median APB)",
    "C(muscle)[T.Ulnar ADM]": "Ulnar ADM (vs Median APB)",
    "C(muscle)[T.Ulnar FDI]": "Ulnar FDI (vs Median APB)",
    "height": "Height (standardised)",
    "height:C(muscle)[T.Median PT]": "Height x Median PT",
    "height:C(muscle)[T.Peroneal TA]": "Height x Peroneal TA",
    "height:C(muscle)[T.Tibial AH]": "Height x Tibial AH",
    "height:C(muscle)[T.Ulnar ADM]": "Height x Ulnar ADM",
    "height:C(muscle)[T.Ulnar FDI]": "Height x Ulnar FDI",
    "C(sex)[T.M]": "sex (male as reference)"
}

reg_summary_df = pd.DataFrame({"coef": ols_model.params, "p_val": ols_model.pvalues})
reg_summary_df["ci_lower"] = ols_model.conf_int()[0]
reg_summary_df["ci_upper"] = ols_model.conf_int()[1]
reg_summary_df.index = reg_summary_df.index.map(name_map)
reg_summary_df.to_csv(f"results/{measure}/regression_output.csv")

In [ ]:
reg_summary_df

In [ ]:
df.height_unadjusted.quantile([0.05,0.95]) # Cohort 90% height range

Q-Q plot of residuals

In [ ]:
plot_df = df.copy()
plot_df["resid"] = ols_model.resid

g = sns.FacetGrid(plot_df, col="muscle", col_wrap=3)

g.map_dataframe(
    lambda data, color: sm.qqplot(
        data["resid"],
        line="45",
        ax=plt.gca()
    )
)

In [ ]:
plot_df["resid"].plot(kind="hist", bins=50)

In [ ]:
plot_df["resid"].skew() # skew / normal distribution. 0 for perfectly normal

Height adjusted limits

In [ ]:
df_reg = pd.read_csv(f"data/{measure}.csv")
height_grid = np.arange(155, 190, 5) # Cohort 90% height range

In [ ]:
# Load in scaler
scaler = joblib.load(f"models/{measure}_scaler.joblib")

height_grid_scaled = scaler.transform(
    pd.DataFrame({"height": height_grid})
)  # Apply the same scaling to new data

In [ ]:
prediction_grid_list=[]
for muscle in ['Median (APB', 'Median PT', 'Ulnar ADM', 'Ulnar FDI', 'Peroneal TA', 'Tibial AH']:
    new_df = pd.DataFrame({"height": height_grid_scaled.reshape(-1), "muscle": muscle})
    prediction_grid_list.append(new_df)
prediction_grid = pd.concat(prediction_grid_list)

Bootstrap estimate of height-adjusted 97.5th percentile

In [ ]:
cmct_string = "value ~ height + C(muscle) + height * C(muscle)"
reg_model, original_predictions, boot_prediction_df = (
    utils.bootstrap_ols_predictions(
        df=df,
        formula=cmct_string,
        prediction_grid=prediction_grid,
        cluster_col="patient_id",
        n_boot=10_000, # 10,000 iterations
        random_state=42,
    )
)

Bootstrap derived confidence intervals

In [ ]:
ols_ci = (
    boot_prediction_df
    .groupby(["height", "muscle"])["upper_limit"]
    .agg(
        ols_lower=lambda x: np.quantile(x, 0.025),
        ols_upper=lambda x: np.quantile(x, 0.975),
    )
    .reset_index()
)

ols_plot = original_predictions.merge(
    ols_ci,
    on=["height", "muscle"],
)

Store height-adjusted limits

In [ ]:
tmp = ols_plot.assign(
    interval=lambda d: (
        d["upper_limit"].round(2).astype(str)
        + " ("
        + d["ols_lower"].round(2).astype(str)
        + ", "
        + d["ols_upper"].round(2).astype(str)
        + ")"
    )
)

wide = tmp.pivot(
    index="height",
    columns="muscle",
    values="interval"
)

In [ ]:
wide

In [ ]:
wide.to_csv(
    f"results/{measure}/height_adjusted_reference_limits.csv", index=True
)